In [ ]:
import pandas as pd
from glob import glob
from typing import Dict, List, Tuple
import re
import random
import xml.etree.ElementTree as ET
from xml.dom import minidom
from tqdm import tqdm
import matplotlib.pyplot as plt
import json
import sys
sys.path.insert(0, "..")
from pipeline.data.preprocessing import preprocess_text
from pipeline.data.xml_utils import convert_to_xml
from pipeline.data.augmentator import *
from pipeline.data.augmentations import *

In [ ]:
DATA_SOURCE = ["hermes"]
PSEUDO_LABEL = False
COMBINE_SAMPLES = True
ADD_FEW_SHOT_EXAMPLES = False

In [ ]:
random.seed(2025)

In [ ]:
all_hermes_df = []
files = glob(f"../data/hermes/hermes_pseudolabeled.csv") if PSEUDO_LABEL else glob(f"../data/hermes/output_*_hermes.csv")
for path in files:
    hermes = pd.read_csv(path)
    hermes.drop(index=hermes[hermes["features_to_include"].isna()].index, inplace=True)
    hermes.drop(index=hermes[hermes["default_response_1"].isna()].index, inplace=True)
    hermes['model'] = "Hermes"
    all_hermes_df.append(hermes)

all_hermes_df = pd.concat(all_hermes_df)
try:
    # original files
    all_hermes_df.drop(columns=["index"], inplace=True)
    faulty_indices = [18765, 19717, 21618, 21362, 21524, 21507, 22274, 22506]
    all_hermes_df.drop(index=faulty_indices, inplace=True)
    all_hermes_df.reset_index(inplace=True, drop=True)
except:
    # pseudolabel files
    pass
all_hermes_df.reset_index(inplace=True, drop=True)

In [ ]:
all_haiku_df = []
files = glob("../data/haiku/haiku_pseudolabeled.csv") if PSEUDO_LABEL else glob("../data/haiku/output_*_haiku.csv")
for path in files:
    haiku = pd.read_csv(path)
    haiku['model'] = "Haiku"
    all_haiku_df.append(haiku)
    
all_haiku_df = pd.concat(all_haiku_df)
try:
    all_haiku_df.rename(columns={"raw_report": "default_response_0", "anonymized_report": "default_response_1"}, inplace=True)
    all_haiku_df['report_length'] = all_haiku_df["criterion"].apply(lambda x: eval(x)["report_length"])
    all_haiku_df['features_to_include'] = all_haiku_df["criterion"].apply(lambda x: eval(x)["features_to_include"])
    all_haiku_df.drop(columns=["criterion"], inplace=True)
except:
    all_haiku_df['features_to_include'] = all_haiku_df['features_to_include'].astype(str)

all_haiku_df.reset_index(inplace=True, drop=True)
all_haiku_df = all_haiku_df[["default_response_0", "default_response_1", "report_length", "features_to_include", "model"]]


In [ ]:
all_deepseek_df = []
files = glob("../data/deepseek/deepseek_pseudolabeled.csv") if PSEUDO_LABEL else glob("../data/deepseek/output_*_deepseek.csv")
for path in files:
    deepseek = pd.read_csv(path)
    deepseek['model'] = "DeepSeek"
    all_deepseek_df.append(deepseek)
    
all_deepseek_df = pd.concat(all_deepseek_df)
try:
    all_deepseek_df.rename(columns={"raw_report": "default_response_0", "anonymized_report": "default_response_1"}, inplace=True)
    all_deepseek_df['report_length'] = all_deepseek_df["criterion"].apply(lambda x: eval(x)["report_length"])
    all_deepseek_df['features_to_include'] = all_deepseek_df["criterion"].apply(lambda x: eval(x)["features_to_include"])
    all_deepseek_df.drop(columns=["criterion"], inplace=True)
except:
    all_deepseek_df['features_to_include'] = all_deepseek_df['features_to_include'].astype(str)

all_deepseek_df.reset_index(inplace=True, drop=True)
all_deepseek_df = all_deepseek_df[["default_response_0", "default_response_1", "report_length", "features_to_include", "model"]]

In [ ]:
print(f"Number of Haiku rows: {len(all_haiku_df)}")
print(f"Number of Hermes rows: {len(all_hermes_df)}")
print(f"Number of DeepSeek rows: {len(all_deepseek_df)}")

In [ ]:
final_dfs = []
if len(all_deepseek_df) > 0 and "deepseek" in DATA_SOURCE:
    final_dfs.append(all_deepseek_df)
if len(all_haiku_df) > 0 and "haiku" in DATA_SOURCE:
    final_dfs.append(all_haiku_df)
if len(all_hermes_df) > 0 and "hermes" in DATA_SOURCE:
    final_dfs.append(all_hermes_df)
final_df = pd.concat(final_dfs)

In [ ]:
words_to_remove = ["<anonymized_report>", "</anonymized_report>", "<raw_report>", "</raw_report>", "<anonymized_report/>", "<raw_report/>", "<anonymized_report/>", "<raw_report/>", "</anonymized_report"]
for word in words_to_remove:
    final_df["default_response_0"] = final_df["default_response_0"].str.replace(word, "")
    final_df["default_response_1"] = final_df["default_response_1"].str.replace(word, "")
    
final_df["default_response_0"] = final_df["default_response_0"].str.strip()
final_df["default_response_1"] = final_df["default_response_1"].str.strip()

final_df.drop(index=final_df[final_df["default_response_0"].isna()].index, inplace=True)
final_df.drop(index=final_df[final_df["default_response_1"].isna()].index, inplace=True)
final_df.drop(index=final_df[final_df["default_response_0"] == ""].index, inplace=True)
final_df.drop(index=final_df[final_df["default_response_1"] == ""].index, inplace=True)

final_df.drop_duplicates(subset=["default_response_1"], inplace=True)
final_df.drop_duplicates(subset=["default_response_0"], inplace=True)
final_df.reset_index(inplace=True, drop=True)

## Fixing incoherent tags issue
We will check if the tags in the `features_to_include` column are present in the `default_response_1` column. If not, we will drop the row

In [ ]:
final_df['features_to_include'] = final_df['features_to_include'].str.replace("hospital", "place")
def varify_tag_presence(text, tags):
    non_present_tags = []
    for tag in tags:
        if f"<{tag}" not in text or f"</{tag}>" not in text:
            non_present_tags.append(tag)
    return non_present_tags

final_df['non_present_tags'] = final_df.apply(lambda row: varify_tag_presence(row["default_response_1"], row["features_to_include"].split(", ")), axis=1)
final_df[final_df["non_present_tags"].apply(len) > 0]

In [ ]:
# Count the occurrences of each number of non-present tags
tag_counts = final_df['non_present_tags'].apply(len).value_counts().sort_index()

# Count the occurrences of each specific non-present tag
tag_names = final_df['non_present_tags'].explode().value_counts()

# Create the bar plot for number of non-present tags
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

tag_counts.plot(kind='bar', ax=ax1)
ax1.set_title('Distribution of Number of Non-Present Tags')
ax1.set_xlabel('Number of Non-Present Tags')
ax1.set_ylabel('Count')

# Add value labels on top of each bar
for i, v in enumerate(tag_counts):
    ax1.text(i, v, str(v), ha='center', va='bottom')

# Create the bar plot for specific non-present tags
tag_names.plot(kind='bar', ax=ax2)
ax2.set_title('Distribution of Specific Non-Present Tags')
ax2.set_xlabel('Tag Name')
ax2.set_ylabel('Count')

# Add value labels on top of each bar
for i, v in enumerate(tag_names):
    ax2.text(i, v, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
# Drop rows with non-present tags
final_df = final_df[final_df['non_present_tags'].apply(lambda x: len(x) == 0)]
final_df = final_df.reset_index(drop=True)
print(f"Number of rows after dropping: {len(final_df)}")

## Preprocessing

In [ ]:
final_df = preprocess_text(final_df, input_col="default_response_0", output_col="default_response_1")

## Creating Composite Reports
We will create composite reports by combining `default_response_1` and `default_response_0` from 5 different rows.

In [ ]:
def update_id_attribute(text, prefix):
    # search for name and place tags and update the id attribute by adding the suffix to the current id
    pattern = r'<(name|place)\s+[^>]*?id="([^"]*)"[^>]*>(.*?)</(name|place)>'
    for match in re.finditer(pattern, text):
        tag = match.group(1)
        id_attr = match.group(2)
        content = match.group(3)
        new_id = f"{prefix}{id_attr}"
        # Preserve original tag attributes by only replacing the id value
        text = text.replace(f'id="{id_attr}"', f'id="{new_id}"')
    return text

In [ ]:
if COMBINE_SAMPLES:
    composite_reports = []
    number_of_composite_reports = len(final_df) // 20 # 5% of the dataset size
    for i in tqdm(range(number_of_composite_reports)):
        random_indices = random.sample(range(len(final_df)), 10)
        random_indices = [index for index in random_indices if index < len(final_df) - 404] # reserved for testing
        composite_response_0 = []
        composite_response_1 = []
        for i, index in enumerate(random_indices):
            composite_response_0.append(update_id_attribute(final_df.iloc[index]["default_response_0"], f"{i}"))
            composite_response_1.append(update_id_attribute(final_df.iloc[index]["default_response_1"], f"{i}"))
        composite_response_0 = "\n\n".join(composite_response_0)
        composite_response_1 = "\n\n".join(composite_response_1)
        composite_reports.append({"default_response_0": composite_response_0, "default_response_1": composite_response_1})
        # remove the rows from the final_df
        for index in random_indices:
            final_df = final_df.drop(index)
            
        final_df.reset_index(inplace=True, drop=True)
        
    composite_df = pd.DataFrame(composite_reports)
    composite_df.head()
    
    last_404_rows = final_df.iloc[-404:]
    rest_of_df = final_df.iloc[:-404]
    final_df = pd.concat([rest_of_df, composite_df])
    final_df.sample(frac=1).reset_index(inplace=True, drop=True)
    final_df = pd.concat([final_df, last_404_rows])
    final_df.reset_index(inplace=True, drop=True)
    print(f"Number of composite reports: {len(composite_df)}")
    print(f"Average Number of words in composite reports: {composite_df['default_response_0'].apply(lambda x: len(x.split())).mean()}")

## Augmenting the dataset
We will augment the dataset by adding variations of dates.

In [ ]:
augmentor = TextAugmentor()
augmentor.add_augmentation(global_sanitize_date, 1)
augmentor.add_augmentation(global_date_random_shift, 1)
augmentor.add_augmentation(global_uniform_date, 0.2)
augmentor.add_augmentation(global_add_time_to_date, 0.1)
augmentor.add_augmentation(global_drop_prepositions, 1)
augmentor.add_augmentation(global_convert_yo, 1)
augmentor.add_augmentation(global_replace_patient_names, 0.5)
augmentor.add_augmentation(global_remove_on_before_date, 0.5)
augmentor.add_augmentation(global_join_paragraphs, 0.2)
augmentor.add_augmentation(global_drop_dot_in_title, 0.2)
augmentor.add_augmentation(global_drop_title_before_name, 0.2)
augmentor.add_augmentation(global_update_ids, 1)
augmentor.add_augmentation(global_remove_phis, 0.02)
augmentor.add_augmentation(global_remove_comma_before_suffixes, 0.5)
date_augmentations = [ # One of
    (ymd_to_mdy, 0.15),
    (ymd_to_dmy, 0.15),
    (ymd_to_mdy2, 0.15),
    (ymd_remove_year, 0.15),
    (ymd_remove_year2, 0.15),
    (ymd_to_wordy, 0.15),
    (ymd_to_wordy2, 0.15),
    (ymd_to_wordy3, 0.15),
    (ymd_remove_day, 0.15),
    (ymd_remove_day_and_month, 0.30),
    (ymd_remove_day_and_year, 0.15),
    (ymd_to_wordy_short_month_no_year, 0.15),
    (ymd_to_wordy_comma_style, 0.15),
    (ymd_to_wordy_no_comma, 0.15),
    (ymd_to_wordy_short_month, 0.15)
]
augmentor.add_augmentation(date_augmentations, 0.95, "date")
augmentor.add_augmentation(fix_lastname_firstname, 0.2, "name")
augmentor.add_augmentation(introduce_misspelling_in_name, 0.1, "name")
phone_augmentations = [
    (add_parantheses_to_phone_number, 0.1),
    (add_dash_to_phone_number, 0.1),
    (add_space_between_phone_number_parts, 0.1),
    (add_international_code_to_phone_number, 0.1)
]
augmentor.add_augmentation(phone_augmentations, 0.95, "phone")
augmentor.add_augmentation(global_remove_spaces_inside_tags, 0.1)
augmentor.add_augmentation(global_make_sentence_uppercase, 0.1)
def augment_dataframe(df, column_name, new_column_name, augmentor: TextAugmentor):
    df[new_column_name] = df[column_name].apply(augmentor.augment)
    return df

augment_dataframe(final_df, "default_response_1", "default_response_1_augmented", augmentor)

## Creating ground truth (v1)
We will now remove the anonymization and use that as our ground truth.

In [ ]:
final_v1 = final_df.copy()

In [ ]:
def remove_anonymization_tags(text):
    # List of all tags to remove
    tags = ['name', 'age', 'date', 'place', 'address', 'phone', 'email', 'id', 'url', 'device']
    
    # Function to remove a specific tag
    def remove_tag(tag, text):
        pattern = rf'<{tag}(?:\s+[^>]*)?>(.*?)</{tag}>'
        return re.sub(pattern, r'\1', text)
    
    # Remove each tag
    for tag in tags:
        text = remove_tag(tag, text)
    
    return text

def handle_formatted_prop(text):
    tags = ['name', 'age', 'date', 'place', 'address', 'phone', 'email', 'id', 'url', 'device']
    
    def process_tag(tag, text):
        pattern = rf'<{tag}(\s+[^>]*?)?(formatted="([^"]*)")?(.*?)>(.*?)</{tag}>'
        
        def replace(match):
            prefix = match.group(1) or ''
            formatted_attr = match.group(2) or ''
            formatted_value = match.group(3)
            other_attrs = match.group(4) or ''
            content = match.group(5)
            
            # If formatted value exists, use it; otherwise, keep the original content
            new_content = formatted_value if formatted_value else content
            
            # Remove the formatted attribute if it exists
            attrs = (prefix + ' ' + other_attrs).strip()
            
            # Reconstruct the tag with the new content
            return f'<{tag}{" " + attrs if attrs else ""}>{new_content}</{tag}>'
        
        return re.sub(pattern, replace, text)
    
    # Process each tag
    for tag in tags:
        text = process_tag(tag, text)
    
    return text

def deanonymize_dataframe_v1(df, column_name, new_column_name):
    df[new_column_name] = df[column_name].apply(remove_anonymization_tags)
    df[column_name] = df[column_name].apply(handle_formatted_prop)
    return df

final_v1 = deanonymize_dataframe_v1(final_v1, "default_response_1_augmented", "default_response_1_deanonymized")

In [ ]:
final_v1 = final_v1[["default_response_1_augmented", "default_response_1_deanonymized"]]
final_v1.columns = ["anonymized", "original"]
final_v1.head()

## Creating ground truth (v2)
We will now remove the anonymization and use that as our ground truth.

In [ ]:
final_v2 = final_df.copy()

In [ ]:
def deanonymize_dataframe_v2(df, column_name, xml_column_name, native_column_name):
    df[xml_column_name] = df[column_name].apply(convert_to_xml)
    df[native_column_name] = df[column_name].apply(remove_anonymization_tags)
    return df

final_v2 = deanonymize_dataframe_v2(final_v2, "default_response_1_augmented", "xml_ground_truth", "native_text")

In [ ]:
final_v2 = final_v2[["xml_ground_truth", "native_text"]]
final_v2.columns = ["anonymized", "original"]
final_v2.head()

## Adding few shot examples
We will add few shot examples to the dataset.

In [ ]:
final_v2['examples'] = "NONE"
if ADD_FEW_SHOT_EXAMPLES:
    num_examples = len(final_v2) // 10 # 10% of the dataset size (~ 2x this will be removed as examples)
    indices_with_examples = random.choices(final_v2.index[:-404], k=num_examples)
    examples_indices = []
    for index in indices_with_examples:
        examples = []
        for _ in range(random.choice([1, 2, 3])): # 1 to 3 examples
            example_idx = None
            while example_idx is None:
                example_idx = random.choice(final_v2.index[:-404])
                if example_idx in indices_with_examples:
                    example_idx = None
                    continue
                
                original = final_v2.at[example_idx, "original"]
                anonymized_example = final_v2.at[example_idx, "anonymized"]
                if len(original.split()) > 500: # too long
                    example_idx = None
                    continue
                anonymized_example = anonymized_example.replace("<phi_entities>\n", "").replace("\n</phi_entities>", "")
                examples.append({"original": original, "anonymized": anonymized_example})
                examples_indices.append(example_idx)
                    
        final_v2.at[index, "examples"] = json.dumps(examples)
    
    # Remove the examples from the dataset
    final_v2.drop(index=examples_indices, inplace=True)
    final_v2.reset_index(inplace=True, drop=True)

In [ ]:
# Shuffle the dataframe while keeping validation set at the end
train_df = final_v2[:-404].copy()
val_df = final_v2[-404:].copy()

# Sort train_df by number of lines in anonymized column
train_df['line_count'] = train_df['anonymized'].str.count('\n') 
train_df = train_df.sort_values('line_count').drop('line_count', axis=1)

# Recombine with validation data
final_v2_lines = pd.concat([train_df, val_df]).reset_index(drop=True)

# Count the number of words in the 'original' column
line_counts = final_v2_lines['anonymized'].apply(lambda x: len(x.split("\n")))

# Create a line plot of word counts
plt.figure(figsize=(10, 6))

# Add light green background for last 404 examples
plt.axvspan(len(line_counts)-404, len(line_counts), color='lightgreen', alpha=0.3)

# Add light red background for lines with 12 lines (empty lines)
plt.axvspan(0, len(line_counts[line_counts == 12]), color='lightcoral', alpha=0.3)

plt.plot(range(len(line_counts)), line_counts, color='skyblue')
plt.title('Line Count Distribution Across Samples')
plt.xlabel('Sample Index')
plt.ylabel('Number of Lines')
plt.grid(True, alpha=0.75)
plt.show()

In [ ]:
# Shuffle the training dataframe
train_df = train_df.sample(frac=1).reset_index(drop=True)
final_v2 = pd.concat([train_df, val_df]).reset_index(drop=True)

In [ ]:
final_v2[final_v2["anonymized"]=="<phi_entities>\n</phi_entities>"]

In [ ]:
print("Final Number of rows: ", len(final_v2))

In [ ]:
final_v1[:-404].to_csv("../data/v1_train.csv", index=False)
final_v1[-404:].to_csv("../data/v1_test.csv", index=False)
final_v2[:-404].to_csv("../data/train.csv", index=False)
final_v2[-404:].to_csv("../data/test.csv", index=False)